# سیستم RAG برای استخراج خواص فیزیکی از مقالات — اجرا در Google Colab

سلول‌ها را **به ترتیب از بالا به پایین** اجرا کنید (دکمه‌ی ▶ کنار هر سلول).

**پیش از شروع:** از منوی *Runtime → Change runtime type* گزینه‌ی **T4 GPU** را انتخاب کنید.

| بخش | کار |
|---|---|
| ۱–۴ | آماده‌سازی (هر بار که کولب باز می‌شود، یک بار) |
| ۵ | افزودن مقاله‌ها و ساخت پایگاه داده‌ی برداری |
| ۶ | پرسش و پاسخ |
| ۷ | ساخت «بسته‌ی منابع» برای آپلود در چت Gemini |
| ۸ | استخراج جدول خواص |

مقاله‌ها و پایگاه داده در **Google Drive** (پوشه‌ی `matrag_data`) ذخیره می‌شوند تا با بسته شدن کولب پاک نشوند.

In [ ]:
#@title ۱. بررسی کارت گرافیک
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ کارت گرافیک فعال نیست: Runtime → Change runtime type → T4 GPU"

In [ ]:
#@title ۲. اتصال به Google Drive و تنظیم مسیرها
import os
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.makedirs(WORK, exist_ok=True)
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
# The vector database runs on Colab's local disk (SQLite is unreliable on Drive)
# and is copied to Drive after each build (step 5).
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
!mkdir -p "{WORK}/storage" /content/storage && cp -rT "{WORK}/storage" /content/storage
print('Data folder:', WORK)

In [ ]:
#@title ۳. نصب پروژه از GitHub
BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -v "already satisfied" | tail -3
!git log --oneline -1

In [ ]:
#@title ۴. راه‌اندازی مدل زبانی
#@markdown **ollama**: مدل متن‌باز روی کارت گرافیک کولب (رایگان، بدون سهمیه). اولین بار دانلود مدل چند دقیقه طول می‌کشد.
#@markdown **gemini**: به کلید API در بخش Secrets کولب (آیکون 🔑 سمت چپ) با نام `MATRAG_GOOGLE_API_KEY` نیاز دارد.
LLM_PROVIDER = "ollama"  #@param ["ollama", "gemini"]
OLLAMA_MODEL = "qwen3:14b"  #@param ["qwen3:14b", "qwen3:8b", "qwen3:4b-instruct-2507-q4_K_M"]
import os, subprocess, time
os.environ['MATRAG_LLM_PROVIDER'] = LLM_PROVIDER

if LLM_PROVIDER == "ollama":
    os.environ['MATRAG_OLLAMA_MODEL'] = OLLAMA_MODEL
    os.environ['MATRAG_OLLAMA_THINKING'] = 'false'  # faster answers from Qwen3
    if subprocess.run('which ollama', shell=True, capture_output=True).returncode != 0:
        !apt-get install -y -qq zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True)
    time.sleep(5)
    !ollama pull {OLLAMA_MODEL} 2>&1 | tail -1
else:
    from google.colab import userdata
    os.environ['MATRAG_GOOGLE_API_KEY'] = userdata.get('MATRAG_GOOGLE_API_KEY')
    print('Gemini key loaded.')

In [ ]:
#@title ۵. افزودن مقاله‌ها و ساخت پایگاه داده
#@markdown فایل‌های PDF را در Google Drive در پوشه‌ی `matrag_data/data/<CORPUS>/pdfs` بگذارید،
#@markdown یا شناسه‌ی arXiv مقاله‌ها را (با کاما جدا) وارد کنید تا خودکار دانلود شوند.
#@markdown مقاله‌هایی که قبلاً پردازش شده‌اند دوباره تبدیل نمی‌شوند.
CORPUS = "hitran"  #@param ["hitran", "optical"]
ARXIV_IDS = "1906.01475, 2602.23812, 2605.11761"  #@param {type:"string"}
import os, pathlib, urllib.request, time
os.environ['MATRAG_CORPUS'] = CORPUS
pdf_dir = pathlib.Path(os.environ['MATRAG_DATA_DIR']) / CORPUS / 'pdfs'
pdf_dir.mkdir(parents=True, exist_ok=True)
for arxiv_id in [a.strip() for a in ARXIV_IDS.split(',') if a.strip()]:
    target = pdf_dir / f'arxiv_{arxiv_id}.pdf'
    if not target.exists():
        req = urllib.request.Request(f'https://arxiv.org/pdf/{arxiv_id}', headers={'User-Agent': 'Mozilla/5.0'})
        target.write_bytes(urllib.request.urlopen(req).read())
        print('downloaded', target.name)
        time.sleep(3)  # be polite to arXiv
!matrag ingest
!matrag info
!cp -rT /content/storage "{WORK}/storage" && echo "Database saved to Drive."

In [ ]:
#@title ۶. پرسش و پاسخ
QUESTION = "How much larger are water-vapor-broadened half-widths of CO2 lines compared to air-broadened ones?"  #@param {type:"string"}
TOP_K = 6  #@param {type:"integer"}
COMPARE_WITHOUT_RAG = True  #@param {type:"boolean"}
print("===== With RAG =====")
!matrag ask "{QUESTION}" --top-k {TOP_K}
if COMPARE_WITHOUT_RAG:
    print("\n===== Without RAG (the model alone) =====")
    !matrag ask "{QUESTION}" --no-rag

In [ ]:
#@title ۷. ساخت «بسته‌ی منابع» برای آپلود در چت Gemini (بدون مصرف API)
QUESTION = "How much larger are water-vapor-broadened half-widths of CO2 lines compared to air-broadened ones?"  #@param {type:"string"}
LANGUAGE = "Persian"  #@param ["Persian", "English"]
TOP_K = 6  #@param {type:"integer"}
!matrag pack "{QUESTION}" --language {LANGUAGE} --top-k {TOP_K} -o /content/pack.md
from google.colab import files
files.download('/content/pack.md')

In [ ]:
#@title ۸. استخراج جدول خواص
PROPERTIES = "air-broadened half-width, self-broadened half-width, line intensity"  #@param {type:"string"}
TOP_K = 4  #@param {type:"integer"}
import os, pandas as pd
props = ' '.join(f'-p "{p.strip()}"' for p in PROPERTIES.split(',') if p.strip())
out = f"{WORK}/results/{os.environ['MATRAG_CORPUS']}_extracted.csv"
!matrag extract {props} --top-k {TOP_K} -o "{out}"
df = pd.read_csv(out)
cols = ['doc_id', 'pages', 'material', 'property', 'value_text', 'unit', 'spectral_position',
        'temperature_K', 'broadener', 'method', 'value_in_source', 'evidence_verified']
df[cols]